In [2]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.data_ingestion import ingest_data

df = ingest_data()

print("Shape:", df.shape)
df.head()

Extracting: multilingual-customer-support-tickets.zip
Loading: aa_dataset-tickets-multi-lang-5-2-50-version.csv

Data ingestion completed.
File: aa_dataset-tickets-multi-lang-5-2-50-version.csv
Shape: (28587, 16)
Shape: (28587, 16)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [3]:
df_en = df[df["language"] == "en"].copy()

df_en["subject"] = df_en["subject"].fillna("")
df_en["body"] = df_en["body"].fillna("")

df_en["text"] = (
    df_en["subject"] + " " + df_en["body"]
).str.strip()

df_en = df_en[df_en["text"] != ""].copy()

print("English tickets:", len(df_en))

print("\nPriority distribution:")
print(df_en["priority"].value_counts())

print("\nPriority percentages:")
print(
    (df_en["priority"].value_counts(normalize=True) * 100)
    .round(2)
)

English tickets: 16338

Priority distribution:
priority
medium    6618
high      6346
low       3374
Name: count, dtype: int64

Priority percentages:
priority
medium    40.51
high      38.84
low       20.65
Name: proportion, dtype: float64


In [4]:
X = df_en["text"]
y = df_en["priority"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExample ticket:")
print(X.iloc[0])

print("\nTarget priority:")
print(y.iloc[0])

X shape: (16338,)
y shape: (16338,)

Example ticket:
Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

Target priority:
high


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 13070
Testing samples: 3268


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

priority_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 3),
    min_df=2,
    max_features=30000,
    sublinear_tf=True
)

X_train_tfidf = priority_tfidf.fit_transform(X_train)
X_test_tfidf = priority_tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (13070, 30000)
Testing TF-IDF shape: (3268, 30000)


In [7]:
from sklearn.linear_model import LogisticRegression

priority_logistic = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

priority_logistic.fit(X_train_tfidf, y_train)

y_pred_logistic = priority_logistic.predict(X_test_tfidf)

print("Priority Logistic Regression training completed.")

Priority Logistic Regression training completed.


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

priority_logistic_accuracy = accuracy_score(
    y_test,
    y_pred_logistic
)

priority_logistic_precision = precision_score(
    y_test,
    y_pred_logistic,
    average="macro",
    zero_division=0
)

priority_logistic_recall = recall_score(
    y_test,
    y_pred_logistic,
    average="macro",
    zero_division=0
)

priority_logistic_f1 = f1_score(
    y_test,
    y_pred_logistic,
    average="macro",
    zero_division=0
)

print("Priority Logistic Regression")
print("----------------------------")
print("Accuracy :", round(priority_logistic_accuracy, 4))
print("Precision:", round(priority_logistic_precision, 4))
print("Recall   :", round(priority_logistic_recall, 4))
print("Macro F1 :", round(priority_logistic_f1, 4))

Priority Logistic Regression
----------------------------
Accuracy : 0.6297
Precision: 0.6172
Recall   : 0.6271
Macro F1 : 0.6209


In [9]:
from sklearn.naive_bayes import MultinomialNB

priority_nb = MultinomialNB()

priority_nb.fit(X_train_tfidf, y_train)

y_pred_nb = priority_nb.predict(X_test_tfidf)

print("Priority Naive Bayes training completed.")

Priority Naive Bayes training completed.


In [10]:
priority_nb_accuracy = accuracy_score(
    y_test,
    y_pred_nb
)

priority_nb_precision = precision_score(
    y_test,
    y_pred_nb,
    average="macro",
    zero_division=0
)

priority_nb_recall = recall_score(
    y_test,
    y_pred_nb,
    average="macro",
    zero_division=0
)

priority_nb_f1 = f1_score(
    y_test,
    y_pred_nb,
    average="macro",
    zero_division=0
)

print("Priority Naive Bayes")
print("--------------------")
print("Accuracy :", round(priority_nb_accuracy, 4))
print("Precision:", round(priority_nb_precision, 4))
print("Recall   :", round(priority_nb_recall, 4))
print("Macro F1 :", round(priority_nb_f1, 4))

Priority Naive Bayes
--------------------
Accuracy : 0.5373
Precision: 0.6542
Recall   : 0.4587
Macro F1 : 0.425


In [11]:
from sklearn.svm import LinearSVC

priority_svm = LinearSVC(
    class_weight="balanced",
    random_state=42
)

priority_svm.fit(X_train_tfidf, y_train)

y_pred_svm = priority_svm.predict(X_test_tfidf)

print("Priority Linear SVM training completed.")

Priority Linear SVM training completed.


In [12]:
priority_svm_accuracy = accuracy_score(
    y_test,
    y_pred_svm
)

priority_svm_precision = precision_score(
    y_test,
    y_pred_svm,
    average="macro",
    zero_division=0
)

priority_svm_recall = recall_score(
    y_test,
    y_pred_svm,
    average="macro",
    zero_division=0
)

priority_svm_f1 = f1_score(
    y_test,
    y_pred_svm,
    average="macro",
    zero_division=0
)

print("Priority Linear SVM")
print("-------------------")
print("Accuracy :", round(priority_svm_accuracy, 4))
print("Precision:", round(priority_svm_precision, 4))
print("Recall   :", round(priority_svm_recall, 4))
print("Macro F1 :", round(priority_svm_f1, 4))

Priority Linear SVM
-------------------
Accuracy : 0.6965
Precision: 0.6884
Recall   : 0.6845
Macro F1 : 0.6863


In [13]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    priority_tfidf,
    MODELS_DIR / "priority_tfidf_vectorizer.pkl"
)

joblib.dump(
    priority_svm,
    MODELS_DIR / "priority_classifier.pkl"
)

print("Priority model saved successfully.")

Priority model saved successfully.


In [14]:
print("Saved model files:")

for file in sorted(MODELS_DIR.glob("*.pkl")):
    print(file.name)

Saved model files:
priority_classifier.pkl
priority_scaler.pkl
priority_tfidf_vectorizer.pkl
priority_vectorizer.pkl
queue_classifier.pkl
queue_tfidf_vectorizer.pkl
tfidf_vectorizer.pkl
